## init

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set all fonts to Arial size 7
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7
})

In [ ]:
from autoadsorbate import Surface
from ase.io import read, write
from ase.visualize import view
import numpy as np
from ase import Atoms
import random
import math

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator


from cft import Manifold
from autoadsorbate.Particle import get_cube_surface_pts, grid_round_cube
from cft.mesh_utils import compute_outward_vertex_normals_quads

from ase.io import read, write
from ase.visualize import view
from autoadsorbate import Fragment
from autoadsorbate.Surf import attach_fragment
from ase.constraints import FixAtoms
import copy

In [4]:
from mace.calculators import mace_mp
clean_calc = mace_mp(model=
                # '/mnt/c/Users/ef/Desktop/tmp/mace-mh-nl-pbe.model',
                '/mnt/c/Users/ef/Desktop/tmp/mace-omat-0-medium.model',
                device='cpu',
                )

/home/ef/venvs/mace_env/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/ef/venvs/mace_env/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.


In [46]:
p = .1
c = 20
scale = [2,2,2]
bulk = read('./TiN.cif')


slab = bulk.copy()*scale
slab.cell[2][2] += c
slab.positions[:,2] += c/2
slab.arrays['fragments'] = np.array([0 for _ in slab])

#make some vacancies
_s = Surface(slab)
n_inds = [atom.index for atom in _s.atoms if atom.symbol =='N' and atom.index in _s.surf_inds]
n_vac = random.sample(n_inds, math.ceil(p * len(n_inds)))
slab = slab[[atom.index for atom in slab if atom.index not in n_vac]]

slab.set_constraint(FixAtoms(indices=[atom.index for atom in slab if atom.position[2] < slab.cell[2][2]*.5]))
slab.rattle(stdev=.2)

view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [47]:
from ase.optimize import BFGS

slab.calc = clean_calc
opt = BFGS(slab, trajectory='relax.xyz', logfile='relax.log')
opt.run(fmax=0.1)

True

In [49]:
view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

## manifold

In [112]:
smiles = [
    'Cl[P+](C)(C)C'
]

f = Fragment(
    smiles[0], to_initialize=1,
    prune_rms_thresh = 0.0001, # same as in rdkit, small numb = more conformers
    )



User requested to_initialize = 1 conformers.
After pruning with 0.0001; len(conformer_trj) = 1 unique conformers are found.


[15:28:00] UFFTYPER: Unrecognized charge state for atom: 1
[15:28:00] UFFTYPER: Unrecognized charge state for atom: 1


In [113]:
m = Manifold(slab,
             precision=1,
             touch_sphere_size =2.5,
             wrap_on='atoms',
             calc = clean_calc)
m.normals*=-1


In [ ]:
f = Fragment('Cl[P+](C)(C)C', to_initialize=1)
for atoms in f.conformers:
    for atom in atoms:
        if atom.symbol =='P':
            atom.symbol ='Al'


User requested to_initialize = 1 conformers.
After pruning with 0.5; len(conformer_trj) = 1 unique conformers are found.


[15:28:14] UFFTYPER: Unrecognized charge state for atom: 1
[15:28:14] UFFTYPER: Unrecognized charge state for atom: 1


In [110]:
m.run_probe_scan(probes=[f])


Scanning probe positions: 100%|██████████| 81/81 [00:29<00:00,  2.72it/s]

energies.shape = (81, 1)


In [111]:
m.write_grid('grd.xyz')

## population

In [ ]:
f = Fragment('Cl[P+](C)(C)C', to_initialize=100)
for atoms in f.conformers:
    for atom in atoms:
        if atom.symbol =='P':
            atom.symbol ='Al'


m.make_fragment_population(
            population_size = 1,
            fragment = f,
            coverage = .25
            )

view(m.surf_population)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [ ]:
for atoms in m.surf_population:


m.evaluate_surf_population()

Calculating energies: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]
